# Cell 1 — Load Dataset & Basic Overview

In [23]:
import pandas as pd
import os


# Đọc dataset
FILE_PATH = "/content/BVBRC_genome_amr.csv"

df = pd.read_csv(
    FILE_PATH,
    dtype={
        "Genome ID": "string"
    }
)

# ------------------------------------------------------------
# Thông tin cơ bản
# ------------------------------------------------------------

print("=" * 70)
print("DATASET OVERVIEW")
print("=" * 70)

print(f"File name       : {os.path.basename(FILE_PATH)}")
print(f"Number of rows  : {df.shape[0]:,}")
print(f"Number of cols  : {df.shape[1]:,}")
print(
    f"Dataset size    : "
    f"{df.memory_usage(deep=True).sum() / 1024**2:.2f} MB"
)

print(f"Genome ID dtype : {df['Genome ID'].dtype}")

# ------------------------------------------------------------
# First rows
# ------------------------------------------------------------

print("\nFirst 5 rows:")
display(df.head())

DATASET OVERVIEW
File name       : BVBRC_genome_amr.csv
Number of rows  : 4,328
Number of cols  : 21
Dataset size    : 3.20 MB
Genome ID dtype : string

First 5 rows:


,Taxon ID,Genome ID,Genome Name,Antibiotic,Resistant Phenotype,Measurement,Measurement Sign,Measurement Value,Measurement Unit,Laboratory Typing Method,...,Laboratory Typing Platform,Vendor,Testing Standard,Testing Standard Year,Computational Method,Computational Method Version,Computational Method Performance,Evidence,Source,PubMed
0,573,573.66266,Klebsiella pneumoniae 26968_6#121,meropenem,Resistant,>=16,>=,16.0,mg/L,MIC,...,NaN,BioMerieux,NaN,NaN,NaN,NaN,NaN,Laboratory Method,NaN,36801013
1,72407,72407.572,Klebsiella pneumoniae subsp. pneumoniae strain...,meropenem,Resistant,>=16,>=,16.0,mg/L,Broth dilution,...,VITEK 2,BioMerieux,CLSI,2010.0,NaN,NaN,NaN,Laboratory Method,NaN,30787414
2,573,573.13209,Klebsiella pneumoniae KPN1219ec,meropenem,Susceptible,<=1,<=,1.0,mg/L,Broth dilution,...,BD Phoenix,NaN,CLSI,2017.0,NaN,NaN,NaN,Laboratory Method,NaN,28512093;28776045
3,1284814,1284814.3,Klebsiella pneumoniae UHKPC 52,meropenem,Susceptible,<=1,<=,1.0,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Laboratory Method,NaN,NaN
4,573,573.14343,Klebsiella pneumoniae KPN717ec,meropenem,Susceptible,<=1,<=,1.0,mg/L,Broth dilution,...,BD Phoenix,NaN,CLSI,2017.0,NaN,NaN,NaN,Laboratory Method,NaN,28512093;28776045


#Cell 2 — Column Names & Data Types

In [24]:
print("COLUMN INFORMATION")

column_info = pd.DataFrame({
    "Column": df.columns,
    "Data Type": df.dtypes.astype(str).values,
    "Non-Null": df.notna().sum().values,
    "Unique Values": df.nunique(dropna=True).values
})

display(column_info)

COLUMN INFORMATION


,Column,Data Type,Non-Null,Unique Values
0,Taxon ID,int64,4328,141
1,Genome ID,string,4328,4270
2,Genome Name,object,4328,4267
3,Antibiotic,object,4328,1
4,Resistant Phenotype,object,4328,2
5,Measurement,object,2762,42
6,Measurement Sign,object,2477,5
7,Measurement Value,float64,2762,20
8,Measurement Unit,object,2698,2
9,Laboratory Typing Method,object,3572,3


#CELL 3 — DUPLICATE & REPEATED GENOME AUDIT

In [25]:


print("DUPLICATE & REPEATED GENOME AUDIT")

# 1. Kiểm tra duplicate toàn bộ dòng
exact_duplicates = df.duplicated().sum()

print(f"Exact duplicate rows : {exact_duplicates:,}")

# 2. Kiểm tra số Genome ID duy nhất
unique_genomes = df["Genome ID"].nunique()

print(f"Total rows           : {len(df):,}")
print(f"Unique Genome IDs    : {unique_genomes:,}")
print(f"Repeated Genome IDs  : {len(df) - unique_genomes:,}")

# 3. Đếm số lần xuất hiện của từng Genome ID
genome_counts = (
    df["Genome ID"]
    .value_counts()
    .rename_axis("Genome ID")
    .reset_index(name="Number of Records")
)

print("\nDistribution of records per Genome ID:")
display(genome_counts["Number of Records"].value_counts().sort_index())

# 4. Hiển thị những Genome ID xuất hiện nhiều hơn 1 lần
repeated_genomes = genome_counts[
    genome_counts["Number of Records"] > 1
]

print("\nRepeated Genome IDs:")
display(repeated_genomes.head(20))

DUPLICATE & REPEATED GENOME AUDIT
Exact duplicate rows : 44
Total rows           : 4,328
Unique Genome IDs    : 4,270
Repeated Genome IDs  : 58

Distribution of records per Genome ID:


,count
Number of Records,
1,4212
2,58



Repeated Genome IDs:


,Genome ID,Number of Records
0,573.46852,2
1,573.46834,2
2,573.46842,2
3,573.15567,2
4,573.24267,2
5,573.46847,2
6,573.46869,2
7,573.46864,2
8,573.15595,2
9,573.24314,2


#Cell 4 — Compare Repeated Genome Records

In [26]:

print("=" * 70)
print("REPEATED GENOME RECORD COMPARISON")
print("=" * 70)

# Lấy các Genome ID xuất hiện nhiều hơn 1 lần
genome_counts = df["Genome ID"].value_counts()

repeated_genome_ids = genome_counts[
    genome_counts > 1
].index

# Các cột quan trọng để so sánh
compare_columns = [
    "Genome ID",
    "Genome Name",
    "Antibiotic",
    "Resistant Phenotype",
    "Measurement",
    "Measurement Sign",
    "Measurement Value",
    "Measurement Unit",
    "Laboratory Typing Method",
    "Laboratory Typing Platform",
    "Vendor",
    "Testing Standard",
    "Testing Standard Year",
    "Evidence",
    "PubMed"
]

# Lọc các Genome ID bị lặp
df_repeated = df[
    df["Genome ID"].isin(repeated_genome_ids)
][compare_columns].copy()

# Sắp xếp để dễ quan sát
df_repeated = df_repeated.sort_values(
    by="Genome ID"
)

print(f"Repeated Genome IDs: {len(repeated_genome_ids):,}")
print(f"Records involved   : {len(df_repeated):,}")

display(df_repeated.head(20))

REPEATED GENOME RECORD COMPARISON
Repeated Genome IDs: 58
Records involved   : 116


,Genome ID,Genome Name,Antibiotic,Resistant Phenotype,Measurement,Measurement Sign,Measurement Value,Measurement Unit,Laboratory Typing Method,Laboratory Typing Platform,Vendor,Testing Standard,Testing Standard Year,Evidence,PubMed
3969,573.15567,Klebsiella pneumoniae strain KSB1_9I,meropenem,Susceptible,NaN,NaN,NaN,NaN,Broth dilution,VITEK 2,BioMerieux,CLSI,NaN,Laboratory Method,28369261
3522,573.15567,Klebsiella pneumoniae strain KSB1_9I,meropenem,Susceptible,NaN,NaN,NaN,NaN,Broth dilution,VITEK 2,BioMerieux,CLSI,NaN,Laboratory Method,28369261
2523,573.15580,Klebsiella pneumoniae strain KSB2_10B,meropenem,Susceptible,NaN,NaN,NaN,NaN,Broth dilution,VITEK 2,BioMerieux,CLSI,NaN,Laboratory Method,29340588
1952,573.15580,Klebsiella pneumoniae strain KSB2_10B,meropenem,Susceptible,NaN,NaN,NaN,NaN,Broth dilution,VITEK 2,BioMerieux,CLSI,NaN,Laboratory Method,28369261
3813,573.15592,Klebsiella pneumoniae strain MSB1_8A,meropenem,Susceptible,NaN,NaN,NaN,NaN,Broth dilution,VITEK 2,BioMerieux,CLSI,NaN,Laboratory Method,28369261
2720,573.15592,Klebsiella pneumoniae strain MSB1_8A,meropenem,Susceptible,NaN,NaN,NaN,NaN,Broth dilution,VITEK 2,BioMerieux,CLSI,NaN,Laboratory Method,28369261
4221,573.15595,Klebsiella pneumoniae strain KSB1_6J,meropenem,Susceptible,NaN,NaN,NaN,NaN,Broth dilution,VITEK 2,BioMerieux,CLSI,NaN,Laboratory Method,28369261
1098,573.15595,Klebsiella pneumoniae strain KSB1_6J,meropenem,Susceptible,NaN,NaN,NaN,NaN,Broth dilution,VITEK 2,BioMerieux,CLSI,NaN,Laboratory Method,29340588
2530,573.15602,Klebsiella pneumoniae strain INF277,meropenem,Resistant,NaN,NaN,NaN,NaN,Broth dilution,VITEK 2,BioMerieux,CLSI,NaN,Laboratory Method,28369261
2173,573.15602,Klebsiella pneumoniae strain INF277,meropenem,Resistant,NaN,NaN,NaN,NaN,Broth dilution,VITEK 2,BioMerieux,CLSI,NaN,Laboratory Method,29340588


#Cell 5 — Check Conflicting Phenotypes per Genome

In [27]:
print("=" * 70)
print("PHENOTYPE CONFLICT AUDIT")
print("=" * 70)

# Đếm số phenotype khác nhau của mỗi Genome ID
phenotype_counts = (
    df.groupby("Genome ID")["Resistant Phenotype"]
      .nunique()
)

# Genome có nhiều hơn 1 phenotype
conflicting_genomes = phenotype_counts[
    phenotype_counts > 1
]

print(f"Total unique Genome IDs        : {df['Genome ID'].nunique():,}")
print(f"Genomes with phenotype conflict: {len(conflicting_genomes):,}")

# Hiển thị chi tiết các genome có conflict
if len(conflicting_genomes) > 0:

    conflict_ids = conflicting_genomes.index

    conflict_records = (
        df[df["Genome ID"].isin(conflict_ids)]
        .sort_values("Genome ID")
    )

    print("\nDetailed conflicting records:")
    display(conflict_records)

else:
    print("\nNo conflicting Resistant/Susceptible labels found.")

PHENOTYPE CONFLICT AUDIT
Total unique Genome IDs        : 4,270
Genomes with phenotype conflict: 0

No conflicting Resistant/Susceptible labels found.


#Cell 6 — Audit Exact Duplicate Rows

In [28]:

# Đánh dấu toàn bộ các dòng thuộc nhóm duplicate
# keep=False giúp hiển thị cả dòng gốc và dòng bị trùng
exact_duplicate_records = df[
    df.duplicated(keep=False)
].copy()

print(f"Total rows in dataset        : {len(df):,}")
print(f"Duplicate rows to be removed : {df.duplicated().sum():,}")
print(f"Records involved in duplicates: {len(exact_duplicate_records):,}")

# Kiểm tra số Genome ID liên quan
duplicate_genome_count = exact_duplicate_records["Genome ID"].nunique()

print(f"Genome IDs involved          : {duplicate_genome_count:,}")

# Hiển thị một số nhóm duplicate
display(
    exact_duplicate_records
    .sort_values("Genome ID")
    .head(20)
)

Total rows in dataset        : 4,328
Duplicate rows to be removed : 44
Records involved in duplicates: 88
Genome IDs involved          : 44


,Taxon ID,Genome ID,Genome Name,Antibiotic,Resistant Phenotype,Measurement,Measurement Sign,Measurement Value,Measurement Unit,Laboratory Typing Method,...,Laboratory Typing Platform,Vendor,Testing Standard,Testing Standard Year,Computational Method,Computational Method Version,Computational Method Performance,Evidence,Source,PubMed
3969,573,573.15567,Klebsiella pneumoniae strain KSB1_9I,meropenem,Susceptible,NaN,NaN,NaN,NaN,Broth dilution,...,VITEK 2,BioMerieux,CLSI,NaN,NaN,NaN,NaN,Laboratory Method,NaN,28369261
3522,573,573.15567,Klebsiella pneumoniae strain KSB1_9I,meropenem,Susceptible,NaN,NaN,NaN,NaN,Broth dilution,...,VITEK 2,BioMerieux,CLSI,NaN,NaN,NaN,NaN,Laboratory Method,NaN,28369261
3813,573,573.15592,Klebsiella pneumoniae strain MSB1_8A,meropenem,Susceptible,NaN,NaN,NaN,NaN,Broth dilution,...,VITEK 2,BioMerieux,CLSI,NaN,NaN,NaN,NaN,Laboratory Method,NaN,28369261
2720,573,573.15592,Klebsiella pneumoniae strain MSB1_8A,meropenem,Susceptible,NaN,NaN,NaN,NaN,Broth dilution,...,VITEK 2,BioMerieux,CLSI,NaN,NaN,NaN,NaN,Laboratory Method,NaN,28369261
2034,573,573.22564,Klebsiella pneumoniae strain SU1,meropenem,Susceptible,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,EUCAST,NaN,NaN,NaN,NaN,Laboratory Method,NaN,30809212
1613,573,573.22564,Klebsiella pneumoniae strain SU1,meropenem,Susceptible,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,EUCAST,NaN,NaN,NaN,NaN,Laboratory Method,NaN,30809212
2750,573,573.46829,Klebsiella pneumoniae KP-101LU,meropenem,Resistant,=64,=,64.0,mg/L,Broth dilution,...,NaN,NaN,EUCAST,NaN,NaN,NaN,NaN,Laboratory Method,NaN,NaN
3829,573,573.46829,Klebsiella pneumoniae KP-101LU,meropenem,Resistant,=64,=,64.0,mg/L,Broth dilution,...,NaN,NaN,EUCAST,NaN,NaN,NaN,NaN,Laboratory Method,NaN,NaN
984,573,573.46831,Klebsiella pneumoniae KP-106LU,meropenem,Resistant,=32,=,32.0,mg/L,Broth dilution,...,NaN,NaN,EUCAST,NaN,NaN,NaN,NaN,Laboratory Method,NaN,NaN
981,573,573.46831,Klebsiella pneumoniae KP-106LU,meropenem,Resistant,=32,=,32.0,mg/L,Broth dilution,...,NaN,NaN,EUCAST,NaN,NaN,NaN,NaN,Laboratory Method,NaN,NaN


#Cell 7 — Create Deduplicated Dataset

In [29]:

# CELL 7 — CREATE DEDUPLICATED & CLEANED DATASET

print("=" * 70)
print("CREATE DEDUPLICATED DATASET")
print("=" * 70)

# 1. Giữ dataset gốc df không thay đổi, loại bỏ exact duplicates
df_deduplicated = df.drop_duplicates().copy()

# 2. Tự động loại bỏ các cột rỗng 100% (5 cột trống)
empty_cols = df_deduplicated.columns[df_deduplicated.isna().all()].tolist()
df_deduplicated = df_deduplicated.drop(columns=empty_cols)

# Thống kê kết quả
rows_removed = len(df) - len(df_deduplicated)

print(f"Original number of records : {len(df):,}")
print(f"Records removed            : {rows_removed:,}")
print(f"Remaining records          : {len(df_deduplicated):,}")
print(f"Completely empty cols dropped: {empty_cols}")
print(f"Remaining columns          : {df_deduplicated.shape[1]}")

print("\nVerification:")
print(f"Exact duplicate rows remaining: {df_deduplicated.duplicated().sum():,}")
print(f"Unique Genome IDs: {df_deduplicated['Genome ID'].nunique():,}")

CREATE DEDUPLICATED DATASET
Original number of records : 4,328
Records removed            : 44
Remaining records          : 4,284
Completely empty cols dropped: ['Laboratory Typing Method Version', 'Computational Method', 'Computational Method Version', 'Computational Method Performance', 'Source']
Remaining columns          : 16

Verification:
Exact duplicate rows remaining: 0
Unique Genome IDs: 4,270


#Cell 8 — Comprehensive Missing Value Audit

In [30]:
# ============================================================
# CELL 8 — COMPREHENSIVE MISSING VALUE AUDIT
# ============================================================

print("=" * 70)
print("COMPREHENSIVE MISSING VALUE AUDIT")
print("=" * 70)

# Tổng số records sau khi loại exact duplicates
total_rows = len(df_deduplicated)

# Tính số lượng missing
missing_count = df_deduplicated.isna().sum()

# Tính tỷ lệ missing
missing_percent = (
    missing_count / total_rows * 100
)

# Tạo bảng tổng hợp
missing_summary = pd.DataFrame({
    "Column": df_deduplicated.columns,
    "Non-Null Count": df_deduplicated.notna().sum().values,
    "Missing Count": missing_count.values,
    "Missing (%)": missing_percent.round(2).values
})

# Sắp xếp theo tỷ lệ missing giảm dần
missing_summary = (
    missing_summary
    .sort_values("Missing (%)", ascending=False)
    .reset_index(drop=True)
)

display(missing_summary)

# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("MISSING VALUE SUMMARY")
print("=" * 70)

print(f"Total records: {total_rows:,}")
print(f"Total columns: {len(df_deduplicated.columns):,}")

print(
    f"Columns with missing values: "
    f"{(missing_count > 0).sum()}"
)

print(
    f"Completely empty columns: "
    f"{(missing_count == total_rows).sum()}"
)

print(
    f"Columns with no missing values: "
    f"{(missing_count == 0).sum()}"
)

COMPREHENSIVE MISSING VALUE AUDIT


,Column,Non-Null Count,Missing Count,Missing (%)
0,Vendor,1421,2863,66.83
1,Testing Standard Year,2212,2072,48.37
2,Measurement Sign,2436,1848,43.14
3,Laboratory Typing Platform,2509,1775,41.43
4,Measurement Unit,2657,1627,37.98
5,Measurement,2721,1563,36.48
6,Measurement Value,2721,1563,36.48
7,Laboratory Typing Method,3529,755,17.62
8,Testing Standard,3533,751,17.53
9,PubMed,3842,442,10.32



MISSING VALUE SUMMARY
Total records: 4,284
Total columns: 16
Columns with missing values: 10
Completely empty columns: 0
Columns with no missing values: 6


#Cell 9 — Missing Value Pattern by Phenotype

In [31]:
# ============================================================
# CELL 9 — MISSING VALUE PATTERN BY RESISTANT PHENOTYPE
# ============================================================

print("=" * 70)
print("MISSING VALUE PATTERN BY RESISTANT PHENOTYPE")
print("=" * 70)

# Các cột dùng để audit missingness, không bao gồm biến mục tiêu
audit_columns = [
    col for col in df_deduplicated.columns
    if col != "Resistant Phenotype"
]

# Tính tỷ lệ missing (%) của từng cột theo từng phenotype
missing_by_phenotype = (
    df_deduplicated[audit_columns]
    .groupby(df_deduplicated["Resistant Phenotype"])
    .agg(lambda x: x.isna().mean() * 100)
    .T
    .round(2)
)

# Nếu có đúng 2 phenotype, tính chênh lệch missingness
if missing_by_phenotype.shape[1] == 2:
    phenotype_cols = missing_by_phenotype.columns.tolist()

    missing_by_phenotype["Difference (%)"] = (
        missing_by_phenotype[phenotype_cols[0]]
        - missing_by_phenotype[phenotype_cols[1]]
    ).abs().round(2)

    missing_by_phenotype = (
        missing_by_phenotype
        .sort_values("Difference (%)", ascending=False)
    )

display(missing_by_phenotype)

MISSING VALUE PATTERN BY RESISTANT PHENOTYPE


Resistant Phenotype,Resistant,Susceptible,Difference (%)
Measurement,17.96,47.75,29.79
Measurement Value,17.96,47.75,29.79
Laboratory Typing Platform,58.77,30.89,27.88
Measurement Unit,21.67,47.90,26.23
PubMed,23.95,2.03,21.92
Laboratory Typing Method,5.93,24.74,18.81
Testing Standard,28.89,10.62,18.27
Measurement Sign,34.81,48.20,13.39
Vendor,59.01,71.58,12.57
Testing Standard Year,49.94,47.41,2.53


#Cell 10 — Column Role Audit

In [32]:
print("=" * 70)
print("COLUMN ROLE AUDIT")
print("=" * 70)

column_roles = pd.DataFrame({
    "Column": [
        "Taxon ID",
        "Genome ID",
        "Genome Name",
        "Antibiotic",
        "Resistant Phenotype",
        "Measurement",
        "Measurement Sign",
        "Measurement Value",
        "Measurement Unit",
        "Laboratory Typing Method",
        "Laboratory Typing Method Version",
        "Laboratory Typing Platform",
        "Vendor",
        "Testing Standard",
        "Testing Standard Year",
        "Computational Method",
        "Computational Method Version",
        "Computational Method Performance",
        "Evidence",
        "Source",
        "PubMed"
    ],

    "Role": [
        "Taxonomic metadata",
        "Sample identifier",
        "Sample identifier",
        "Experimental context",
        "Target label",
        "Phenotype measurement metadata",
        "Phenotype measurement metadata",
        "Phenotype measurement metadata",
        "Phenotype measurement metadata",
        "Laboratory metadata",
        "Empty column",
        "Laboratory metadata",
        "Laboratory metadata",
        "Laboratory metadata",
        "Laboratory metadata",
        "Empty column",
        "Empty column",
        "Empty column",
        "Evidence metadata",
        "Empty column",
        "Reference metadata"
    ],

    "ML Usage": [
        "Not used as ML feature",
        "Used for genome retrieval / grouping",
        "Reference only",
        "Not used as ML feature",
        "TARGET",
        "Potential leakage - not ML feature",
        "Potential leakage - not ML feature",
        "Potential leakage - not ML feature",
        "Potential leakage - not ML feature",
        "Potential leakage / metadata",
        "Remove",
        "Potential leakage / metadata",
        "Potential leakage / metadata",
        "Potential leakage / metadata",
        "Potential leakage / metadata",
        "Remove",
        "Remove",
        "Remove",
        "Audit/reference only",
        "Remove",
        "Reference only"
    ]
})

display(column_roles)

COLUMN ROLE AUDIT


,Column,Role,ML Usage
0,Taxon ID,Taxonomic metadata,Not used as ML feature
1,Genome ID,Sample identifier,Used for genome retrieval / grouping
2,Genome Name,Sample identifier,Reference only
3,Antibiotic,Experimental context,Not used as ML feature
4,Resistant Phenotype,Target label,TARGET
5,Measurement,Phenotype measurement metadata,Potential leakage - not ML feature
6,Measurement Sign,Phenotype measurement metadata,Potential leakage - not ML feature
7,Measurement Value,Phenotype measurement metadata,Potential leakage - not ML feature
8,Measurement Unit,Phenotype measurement metadata,Potential leakage - not ML feature
9,Laboratory Typing Method,Laboratory metadata,Potential leakage / metadata


#Cell 11 — Target Distribution Audit

In [33]:
print("=" * 70)
print("TARGET DISTRIBUTION AUDIT")
print("=" * 70)

# Đếm số lượng mẫu của từng phenotype
target_counts = (
    df_deduplicated["Resistant Phenotype"]
    .value_counts()
)

# Tính tỷ lệ phần trăm
target_percentages = (
    df_deduplicated["Resistant Phenotype"]
    .value_counts(normalize=True)
    * 100
).round(2)

# Tạo bảng tổng hợp
target_summary = pd.DataFrame({
    "Count": target_counts,
    "Percentage (%)": target_percentages
})

display(target_summary)

# Kiểm tra tỷ lệ imbalance
majority_class = target_counts.max()
minority_class = target_counts.min()

imbalance_ratio = majority_class / minority_class

print("\n" + "=" * 70)
print("CLASS BALANCE SUMMARY")
print("=" * 70)

print(f"Majority class count : {majority_class:,}")
print(f"Minority class count : {minority_class:,}")
print(f"Imbalance ratio      : {imbalance_ratio:.2f}:1")

TARGET DISTRIBUTION AUDIT


,Count,Percentage (%)
Resistant Phenotype,,
Susceptible,2664,62.18
Resistant,1620,37.82



CLASS BALANCE SUMMARY
Majority class count : 2,664
Minority class count : 1,620
Imbalance ratio      : 1.64:1


#Cell 12 — Unique Genome-Level Target Distribution

In [34]:
# ============================================================
# CELL 12 — UNIQUE GENOME-LEVEL TARGET DISTRIBUTION (OPTIMIZED)
# ============================================================

print("=" * 70)
print("UNIQUE GENOME-LEVEL TARGET DISTRIBUTION")
print("=" * 70)

# Tạo cột tạm để đếm số giá trị hợp lệ (non-null) trên mỗi dòng
df_genome_level = (
    df_deduplicated
    .assign(
        non_null_count=df_deduplicated.notna().sum(axis=1)
    )
    .sort_values(
        ["Genome ID", "non_null_count"],
        ascending=[True, False]
    )
    .drop_duplicates(
        subset=["Genome ID"],
        keep="first"
    )
    .drop(columns=["non_null_count"])
    .reset_index(drop=True)
)

# Đếm số genome theo từng phenotype
genome_target_counts = (
    df_genome_level["Resistant Phenotype"]
    .value_counts()
)

# Tính tỷ lệ phần trăm
genome_target_percentages = (
    df_genome_level["Resistant Phenotype"]
    .value_counts(normalize=True)
    * 100
).round(2)

# Tạo bảng tổng hợp
genome_target_summary = pd.DataFrame({
    "Genome Count": genome_target_counts,
    "Percentage (%)": genome_target_percentages
})

display(genome_target_summary)

# ============================================================
# CLASS BALANCE SUMMARY
# ============================================================

majority_class = genome_target_counts.max()
minority_class = genome_target_counts.min()

imbalance_ratio = majority_class / minority_class

print("\n" + "=" * 70)
print("GENOME-LEVEL CLASS BALANCE SUMMARY")
print("=" * 70)

print(f"Total unique genomes : {len(df_genome_level):,}")
print(f"Majority class count : {majority_class:,}")
print(f"Minority class count : {minority_class:,}")
print(f"Imbalance ratio      : {imbalance_ratio:.2f}:1")

UNIQUE GENOME-LEVEL TARGET DISTRIBUTION


,Genome Count,Percentage (%)
Resistant Phenotype,,
Susceptible,2661,62.32
Resistant,1609,37.68



GENOME-LEVEL CLASS BALANCE SUMMARY
Total unique genomes : 4,270
Majority class count : 2,661
Minority class count : 1,609
Imbalance ratio      : 1.65:1


#Cell 13 — Taxonomic Distribution Audit

In [35]:

print("=" * 70)
print("TAXONOMIC DISTRIBUTION AUDIT")
print("=" * 70)

# Đếm số genome theo Taxon ID
taxon_counts = (
    df_genome_level["Taxon ID"]
    .value_counts()
)

# Tính tỷ lệ phần trăm
taxon_percentages = (
    df_genome_level["Taxon ID"]
    .value_counts(normalize=True)
    * 100
).round(2)

# Tạo bảng tổng hợp
taxon_summary = pd.DataFrame({
    "Genome Count": taxon_counts,
    "Percentage (%)": taxon_percentages
})

print(f"Total unique Taxon IDs: {df_genome_level['Taxon ID'].nunique():,}")

print("\nTop 20 Taxon IDs:")
display(taxon_summary.head(20))

TAXONOMIC DISTRIBUTION AUDIT
Total unique Taxon IDs: 141

Top 20 Taxon IDs:


,Genome Count,Percentage (%)
Taxon ID,,
573,3995,93.56
72407,134,3.14
574,3,0.07
1284788,1,0.02
1284787,1,0.02
1284792,1,0.02
1284793,1,0.02
1284794,1,0.02
1284795,1,0.02


#Cell 14 — Taxon Name Verification

In [36]:

print("=" * 70)
print("TAXON NAME VERIFICATION")
print("=" * 70)

# Lấy các Taxon ID phổ biến nhất
top_taxon_ids = (
    df_genome_level["Taxon ID"]
    .value_counts()
    .head(10)
    .index
)

# Hiển thị Genome Name đại diện cho từng Taxon ID
taxon_name_examples = (
    df_genome_level[
        df_genome_level["Taxon ID"].isin(top_taxon_ids)
    ]
    .groupby("Taxon ID")["Genome Name"]
    .agg(
        Genome_Count="count",
        Example_Genome_Names=lambda x: list(x.dropna().unique()[:5])
    )
    .reset_index()
)

display(taxon_name_examples)

TAXON NAME VERIFICATION


,Taxon ID,Genome_Count,Example_Genome_Names
0,573,3995,"[Klebsiella pneumoniae strain AR_0126, Klebsie..."
1,574,3,[Klebsiella pneumoniae subsp. ozaenae strain A...
2,72407,134,[Klebsiella pneumoniae subsp. pneumoniae MRSN7...
3,1284787,1,[Klebsiella pneumoniae UHKPC40]
4,1284788,1,[Klebsiella pneumoniae UHKPC23]
5,1284792,1,[Klebsiella pneumoniae UHKPC67]
6,1284793,1,[Klebsiella pneumoniae UHKPC69]
7,1284794,1,[Klebsiella pneumoniae UHKPC77]
8,1284795,1,[Klebsiella pneumoniae UHKPC96]
9,1284796,1,[Klebsiella pneumoniae DMC0526]


#Cell 15 — Genome Name Consistency Audit

In [37]:

print("=" * 70)
print("GENOME NAME CONSISTENCY AUDIT")
print("=" * 70)

# Kiểm tra genome name có chứa "Klebsiella pneumoniae"
is_kp = (
    df_genome_level["Genome Name"]
    .str.contains(
        "Klebsiella pneumoniae",
        case=False,
        na=False
    )
)

total_genomes = len(df_genome_level)
kp_genomes = is_kp.sum()
non_kp_genomes = (~is_kp).sum()

print(f"Total unique genomes                 : {total_genomes:,}")
print(f'Genomes containing "Klebsiella pneumoniae": {kp_genomes:,}')
print(f"Other / inconsistent genome names    : {non_kp_genomes:,}")

# Hiển thị các genome không chứa tên Klebsiella pneumoniae
if non_kp_genomes > 0:

    print("\nPotentially inconsistent genome names:")

    display(
        df_genome_level.loc[
            ~is_kp,
            [
                "Taxon ID",
                "Genome ID",
                "Genome Name",
                "Resistant Phenotype"
            ]
        ].head(50)
    )

else:
    print("\nAll genome names are consistent with Klebsiella pneumoniae.")

GENOME NAME CONSISTENCY AUDIT
Total unique genomes                 : 4,270
Genomes containing "Klebsiella pneumoniae": 4,270
Other / inconsistent genome names    : 0

All genome names are consistent with Klebsiella pneumoniae.


#Cell 16 — Genome Name Duplicate / Identifier Consistency Audit

In [38]:
# ============================================================
# CELL 16 — IDENTIFIER CONSISTENCY AUDIT
# ============================================================

print("=" * 70)
print("IDENTIFIER CONSISTENCY AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Genome ID -> Number of unique Genome Names
# ------------------------------------------------------------

id_to_name = (
    df_genome_level
    .groupby("Genome ID")["Genome Name"]
    .nunique()
)

ids_with_multiple_names = id_to_name[id_to_name > 1]

print(
    f"Genome IDs with multiple Genome Names: "
    f"{len(ids_with_multiple_names):,}"
)

# ------------------------------------------------------------
# 2. Genome Name -> Number of unique Genome IDs
# ------------------------------------------------------------

name_to_id = (
    df_genome_level
    .groupby("Genome Name")["Genome ID"]
    .nunique()
)

names_with_multiple_ids = name_to_id[name_to_id > 1]

print(
    f"Genome Names with multiple Genome IDs: "
    f"{len(names_with_multiple_ids):,}"
)

# ------------------------------------------------------------
# 3. Summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("IDENTIFIER SUMMARY")
print("=" * 70)

print(f"Unique Genome IDs   : {df_genome_level['Genome ID'].nunique():,}")
print(f"Unique Genome Names : {df_genome_level['Genome Name'].nunique():,}")

# ------------------------------------------------------------
# 4. Show potential inconsistencies
# ------------------------------------------------------------

if len(ids_with_multiple_names) > 0:

    print("\nGenome IDs associated with multiple names:")

    display(
        df_genome_level[
            df_genome_level["Genome ID"].isin(ids_with_multiple_names.index)
        ][
            ["Genome ID", "Genome Name", "Resistant Phenotype"]
        ].sort_values("Genome ID")
    )

if len(names_with_multiple_ids) > 0:

    print("\nGenome Names associated with multiple Genome IDs:")

    display(
        df_genome_level[
            df_genome_level["Genome Name"].isin(names_with_multiple_ids.index)
        ][
            ["Genome ID", "Genome Name", "Resistant Phenotype"]
        ].sort_values("Genome Name")
    )

if len(ids_with_multiple_names) == 0 and len(names_with_multiple_ids) == 0:
    print("\nNo identifier inconsistencies detected.")

IDENTIFIER CONSISTENCY AUDIT
Genome IDs with multiple Genome Names: 0
Genome Names with multiple Genome IDs: 3

IDENTIFIER SUMMARY
Unique Genome IDs   : 4,270
Unique Genome Names : 4,267

Genome Names associated with multiple Genome IDs:


,Genome ID,Genome Name,Resistant Phenotype
1769,573.15217,Klebsiella pneumoniae strain AR_0112,Resistant
1787,573.15420,Klebsiella pneumoniae strain AR_0112,Resistant
1778,573.15369,Klebsiella pneumoniae strain INF158,Resistant
1975,573.15629,Klebsiella pneumoniae strain INF158,Resistant
1770,573.15329,Klebsiella pneumoniae strain KSB1_5D,Resistant
1951,573.15603,Klebsiella pneumoniae strain KSB1_5D,Resistant


#Cell 17 — Duplicate Genome Name Deep Audit

In [39]:
print("=" * 70)
print("DUPLICATE GENOME NAME DEEP AUDIT")
print("=" * 70)

# Tìm Genome Name xuất hiện với nhiều Genome ID
duplicate_names = (
    df_genome_level
    .groupby("Genome Name")["Genome ID"]
    .nunique()
)

duplicate_names = duplicate_names[
    duplicate_names > 1
].index

print(f"Genome Names with multiple Genome IDs: {len(duplicate_names)}")

# Hiển thị toàn bộ metadata của các trường hợp này
duplicate_name_records = (
    df_genome_level[
        df_genome_level["Genome Name"].isin(duplicate_names)
    ]
    .sort_values(["Genome Name", "Genome ID"])
)

display(duplicate_name_records)

DUPLICATE GENOME NAME DEEP AUDIT
Genome Names with multiple Genome IDs: 3


,Taxon ID,Genome ID,Genome Name,Antibiotic,Resistant Phenotype,Measurement,Measurement Sign,Measurement Value,Measurement Unit,Laboratory Typing Method,Laboratory Typing Platform,Vendor,Testing Standard,Testing Standard Year,Evidence,PubMed
1769,573,573.15217,Klebsiella pneumoniae strain AR_0112,meropenem,Resistant,>8,>,8.0,mg/L,Broth dilution,NaN,NaN,CLSI,NaN,Laboratory Method,NaN
1787,573,573.15420,Klebsiella pneumoniae strain AR_0112,meropenem,Resistant,>8,>,8.0,mg/L,Broth dilution,NaN,NaN,CLSI,NaN,Laboratory Method,NaN
1778,573,573.15369,Klebsiella pneumoniae strain INF158,meropenem,Resistant,NaN,NaN,NaN,NaN,Broth dilution,VITEK 2,BioMerieux,CLSI,NaN,Laboratory Method,29340588
1975,573,573.15629,Klebsiella pneumoniae strain INF158,meropenem,Resistant,NaN,NaN,NaN,NaN,Broth dilution,VITEK 2,BioMerieux,CLSI,NaN,Laboratory Method,28369261
1770,573,573.15329,Klebsiella pneumoniae strain KSB1_5D,meropenem,Resistant,NaN,NaN,NaN,NaN,Broth dilution,VITEK 2,BioMerieux,CLSI,NaN,Laboratory Method,29340588
1951,573,573.15603,Klebsiella pneumoniae strain KSB1_5D,meropenem,Resistant,NaN,NaN,NaN,NaN,Broth dilution,VITEK 2,BioMerieux,CLSI,NaN,Laboratory Method,28369261


#Cell 18 — Genome ID Format and Retrieval Readiness Audit

In [40]:
print("=" * 70)
print("GENOME ID FORMAT & RETRIEVAL READINESS AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Kiểm tra dtype
# ------------------------------------------------------------

print(f"Current Genome ID dtype: {df_genome_level['Genome ID'].dtype}")


# ------------------------------------------------------------
# 2. Hiển thị một số Genome ID
# ------------------------------------------------------------

print("\nSample Genome IDs:")

display(
    df_genome_level[
        [
            "Genome ID",
            "Genome Name",
            "Resistant Phenotype"
        ]
    ].head(20)
)


# ------------------------------------------------------------
# ------------------------------------------------------------
# 3. Genome ID dưới dạng string
# ------------------------------------------------------------

df_genome_level["Genome ID String"] = (
    df_genome_level["Genome ID"]
    .astype("string")
    .str.strip()
)

genome_id_as_string = df_genome_level["Genome ID String"]


# ------------------------------------------------------------
# 4. Kiểm tra định dạng Genome ID
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GENOME ID FORMAT CHECK")
print("=" * 70)

decimal_count = (
    genome_id_as_string
    .str.contains(
        r"\.",
        regex=True,
        na=False
    )
    .sum()
)

scientific_count = (
    genome_id_as_string
    .str.contains(
        r"[eE][+-]?\d+",
        regex=True,
        na=False
    )
    .sum()
)

print(f"Genome IDs containing decimal point      : {decimal_count:,}")
print(f"Genome IDs containing scientific notation: {scientific_count:,}")


# ------------------------------------------------------------
# 5. Kiểm tra missing và uniqueness
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GENOME ID INTEGRITY CHECK")
print("=" * 70)

print(f"Total records         : {len(df_genome_level):,}")
print(f"Unique Genome IDs     : {genome_id_as_string.nunique():,}")
print(f"Missing Genome IDs    : {genome_id_as_string.isna().sum():,}")


# ------------------------------------------------------------
# 6. Kiểm tra ví dụ Genome ID chuẩn bị cho retrieval
# ------------------------------------------------------------

print("\nExample Genome IDs prepared for retrieval:")

preview = df_genome_level[
    [
        "Genome ID",
        "Genome Name"
    ]
].copy()

preview["Genome ID String"] = genome_id_as_string

display(
    preview[
        [
            "Genome ID",
            "Genome ID String",
            "Genome Name"
        ]
    ].head(20)
)


print("\nRetrieval readiness audit completed.")

GENOME ID FORMAT & RETRIEVAL READINESS AUDIT
Current Genome ID dtype: string

Sample Genome IDs:


,Genome ID,Genome Name,Resistant Phenotype
0,1284787.3,Klebsiella pneumoniae UHKPC40,Resistant
1,1284788.3,Klebsiella pneumoniae UHKPC23,Resistant
2,1284789.4,Klebsiella pneumoniae UHKPC28,Resistant
3,1284790.3,Klebsiella pneumoniae UHKPC05,Resistant
4,1284791.4,Klebsiella pneumoniae UHKPC47,Resistant
5,1284792.4,Klebsiella pneumoniae UHKPC67,Resistant
6,1284793.4,Klebsiella pneumoniae UHKPC69,Resistant
7,1284794.4,Klebsiella pneumoniae UHKPC77,Resistant
8,1284795.4,Klebsiella pneumoniae UHKPC96,Resistant
9,1284796.3,Klebsiella pneumoniae DMC0526,Resistant



GENOME ID FORMAT CHECK
Genome IDs containing decimal point      : 4,270
Genome IDs containing scientific notation: 0

GENOME ID INTEGRITY CHECK
Total records         : 4,270
Unique Genome IDs     : 4,270
Missing Genome IDs    : 0

Example Genome IDs prepared for retrieval:


,Genome ID,Genome ID String,Genome Name
0,1284787.3,1284787.3,Klebsiella pneumoniae UHKPC40
1,1284788.3,1284788.3,Klebsiella pneumoniae UHKPC23
2,1284789.4,1284789.4,Klebsiella pneumoniae UHKPC28
3,1284790.3,1284790.3,Klebsiella pneumoniae UHKPC05
4,1284791.4,1284791.4,Klebsiella pneumoniae UHKPC47
5,1284792.4,1284792.4,Klebsiella pneumoniae UHKPC67
6,1284793.4,1284793.4,Klebsiella pneumoniae UHKPC69
7,1284794.4,1284794.4,Klebsiella pneumoniae UHKPC77
8,1284795.4,1284795.4,Klebsiella pneumoniae UHKPC96
9,1284796.3,1284796.3,Klebsiella pneumoniae DMC0526



Retrieval readiness audit completed.


#Cell 19 — Genome ID Precision & Collision Audit

In [41]:
print("=" * 70)
print("GENOME ID PRECISION & COLLISION AUDIT")
print("=" * 70)

# 1. Kiểm tra uniqueness của Genome ID hiện tại

total_rows = len(df_genome_level)

unique_genome_ids = (
    df_genome_level["Genome ID"]
    .nunique()
)

print(f"Total genome-level records      : {total_rows:,}")
print(f"Unique Genome IDs        : {unique_numeric_ids:,}")

# 2. Kiểm tra scientific notation trong representation

default_string_ids = (
    df_genome_level["Genome ID"]
    .astype(str)
)

scientific_notation_count = (
    default_string_ids
    .str.contains(r"[eE][+-]?\d+", regex=True)
    .sum()
)

print(
    f"Scientific notation after astype(str): "
    f"{scientific_notation_count:,}"
)

# ------------------------------------------------------------
# 3. Kiểm tra số chữ số sau dấu thập phân
# ------------------------------------------------------------

decimal_lengths = (
    default_string_ids
    .str.split(".")
    .str[1]
    .str.len()
)

print("\nDistribution of decimal lengths:")
display(
    decimal_lengths
    .value_counts()
    .sort_index()
    .rename("Number of Genome IDs")
    .to_frame()
)

# ------------------------------------------------------------
# 4. Hiển thị các ID có trailing zero bị mất khả năng cao
# ------------------------------------------------------------

print("\nSample Genome IDs by decimal length:")

for length in sorted(decimal_lengths.dropna().unique()):

    sample_ids = (
        df_genome_level.loc[
            decimal_lengths == length,
            ["Genome ID", "Genome Name"]
        ]
        .head(5)
    )

    print(f"\n--- Decimal length = {length} ---")
    display(sample_ids)

print("\nPrecision audit completed.")

GENOME ID PRECISION & COLLISION AUDIT
Total genome-level records      : 4,270
Unique Genome IDs        : 4,270
Scientific notation after astype(str): 0

Distribution of decimal lengths:


,Number of Genome IDs
Genome ID,
1,138
2,3
3,121
4,169
5,3839



Sample Genome IDs by decimal length:

--- Decimal length = 1 ---


,Genome ID,Genome Name
0,1284787.3,Klebsiella pneumoniae UHKPC40
1,1284788.3,Klebsiella pneumoniae UHKPC23
2,1284789.4,Klebsiella pneumoniae UHKPC28
3,1284790.3,Klebsiella pneumoniae UHKPC05
4,1284791.4,Klebsiella pneumoniae UHKPC47



--- Decimal length = 2 ---


...


--- Decimal length = 4 ---


,Genome ID,Genome Name
1976,573.1719,Klebsiella pneumoniae strain KpVA-1
1977,573.1720,Klebsiella pneumoniae strain KpVA-2
1978,573.1721,Klebsiella pneumoniae strain KpVA-3
1979,573.1722,Klebsiella pneumoniae strain KpVA-4
1980,573.1723,Klebsiella pneumoniae strain KpVA-5



--- Decimal length = 5 ---


,Genome ID,Genome Name
138,573.12771,Klebsiella pneumoniae strain AR_0126
139,573.12772,Klebsiella pneumoniae strain AR_0113
140,573.12773,Klebsiella pneumoniae strain AR_0138
141,573.12777,Klebsiella pneumoniae strain AR_0120
142,573.12778,Klebsiella pneumoniae strain AR_0125



Precision audit completed.


# CELL 20 — CREATE FINAL GENOME MANIFEST

In [42]:
# ============================================================
# CELL 20 — CREATE FINAL GENOME MANIFEST
# ============================================================

print("=" * 70)
print("CREATE FINAL GENOME MANIFEST")
print("=" * 70)

# ------------------------------------------------------------
# 1. Kiểm tra dataset genome-level
# ------------------------------------------------------------

print(f"Genome-level records before manifest: {len(df_genome_level):,}")
print(f"Unique Genome IDs                  : {df_genome_level['Genome ID'].nunique():,}")

# ------------------------------------------------------------
# 2. Chọn các cột cần thiết cho Genome Manifest
# ------------------------------------------------------------

manifest_columns = [
    "Genome ID",
    "Genome ID String",
    "Genome Name",
    "Taxon ID",
    "Antibiotic",
    "Resistant Phenotype"
]

# Kiểm tra các cột có tồn tại không
missing_columns = [
    col for col in manifest_columns
    if col not in df_genome_level.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

# ------------------------------------------------------------
# 3. Tạo Genome Manifest
# ------------------------------------------------------------

df_genome_manifest = (
    df_genome_level[manifest_columns]
    .copy()
    .sort_values("Genome ID String")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Kiểm tra integrity của Manifest
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GENOME MANIFEST INTEGRITY CHECK")
print("=" * 70)

print(f"Total records             : {len(df_genome_manifest):,}")
print(
    f"Unique Genome ID Strings  : "
    f"{df_genome_manifest['Genome ID String'].nunique():,}"
)
print(
    f"Missing Genome ID Strings : "
    f"{df_genome_manifest['Genome ID String'].isna().sum():,}"
)
print(
    f"Missing Genome Names      : "
    f"{df_genome_manifest['Genome Name'].isna().sum():,}"
)
print(
    f"Missing Target Labels     : "
    f"{df_genome_manifest['Resistant Phenotype'].isna().sum():,}"
)

# ------------------------------------------------------------
# 5. Kiểm tra duplicate Genome IDs
# ------------------------------------------------------------

duplicate_genome_ids = (
    df_genome_manifest["Genome ID String"]
    .duplicated()
    .sum()
)

print(
    f"Duplicate Genome IDs      : "
    f"{duplicate_genome_ids:,}"
)

# ------------------------------------------------------------
# 6. Kiểm tra target distribution
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("GENOME MANIFEST TARGET DISTRIBUTION")
print("=" * 70)

target_distribution = (
    df_genome_manifest["Resistant Phenotype"]
    .value_counts()
)

target_percentage = (
    df_genome_manifest["Resistant Phenotype"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

target_summary = pd.DataFrame({
    "Genome Count": target_distribution,
    "Percentage (%)": target_percentage
})

display(target_summary)

# ------------------------------------------------------------
# 7. Hiển thị mẫu Genome Manifest
# ------------------------------------------------------------

print("\nSample Genome Manifest:")

display(df_genome_manifest.head(20))

print("\n" + "=" * 70)
print("GENOME MANIFEST CREATED SUCCESSFULLY")
print("=" * 70)
print(
    "This manifest is ready for genome sequence retrieval "
    "in File 02."
)

CREATE FINAL GENOME MANIFEST
Genome-level records before manifest: 4,270
Unique Genome IDs                  : 4,270

GENOME MANIFEST INTEGRITY CHECK
Total records             : 4,270
Unique Genome ID Strings  : 4,270
Missing Genome ID Strings : 0
Missing Genome Names      : 0
Missing Target Labels     : 0
Duplicate Genome IDs      : 0

GENOME MANIFEST TARGET DISTRIBUTION


,Genome Count,Percentage (%)
Resistant Phenotype,,
Susceptible,2661,62.32
Resistant,1609,37.68



Sample Genome Manifest:


,Genome ID,Genome ID String,Genome Name,Taxon ID,Antibiotic,Resistant Phenotype
0,1284787.3,1284787.3,Klebsiella pneumoniae UHKPC40,1284787,meropenem,Resistant
1,1284788.3,1284788.3,Klebsiella pneumoniae UHKPC23,1284788,meropenem,Resistant
2,1284789.4,1284789.4,Klebsiella pneumoniae UHKPC28,1284789,meropenem,Resistant
3,1284790.3,1284790.3,Klebsiella pneumoniae UHKPC05,1284790,meropenem,Resistant
4,1284791.4,1284791.4,Klebsiella pneumoniae UHKPC47,1284791,meropenem,Resistant
5,1284792.4,1284792.4,Klebsiella pneumoniae UHKPC67,1284792,meropenem,Resistant
6,1284793.4,1284793.4,Klebsiella pneumoniae UHKPC69,1284793,meropenem,Resistant
7,1284794.4,1284794.4,Klebsiella pneumoniae UHKPC77,1284794,meropenem,Resistant
8,1284795.4,1284795.4,Klebsiella pneumoniae UHKPC96,1284795,meropenem,Resistant
9,1284796.3,1284796.3,Klebsiella pneumoniae DMC0526,1284796,meropenem,Resistant



GENOME MANIFEST CREATED SUCCESSFULLY
This manifest is ready for genome sequence retrieval in File 02.


#CELL 21 — FINAL DATA INTEGRITY ASSERTIONS

In [43]:
# ============================================================
# FINAL DATA INTEGRITY ASSERTIONS
# ============================================================

assert df_deduplicated.duplicated().sum() == 0, \
    "Exact duplicates still exist."

assert df_genome_level["Genome ID"].is_unique, \
    "Genome-level dataset contains duplicated Genome IDs."

assert df_genome_level["Genome ID"].notna().all(), \
    "Missing Genome ID detected."

assert df_genome_level["Resistant Phenotype"].notna().all(), \
    "Missing target label detected."

assert set(df_genome_level["Resistant Phenotype"].unique()) == {
    "Resistant",
    "Susceptible"
}, "Unexpected phenotype labels detected."

assert (
    df_genome_manifest["Genome ID String"].is_unique
), "Duplicate Genome IDs detected in manifest."

assert (
    df_genome_manifest["Genome ID String"].notna().all()
), "Missing Genome ID in manifest."

assert (
    df_genome_manifest["Antibiotic"]
    .str.lower()
    .eq("meropenem")
    .all()
), "Non-meropenem records detected."

print("✓ All final integrity checks passed.")
print(f"✓ Final genomes: {len(df_genome_manifest):,}")

✓ All final integrity checks passed.
✓ Final genomes: 4,270


#Cell 22 — Export Final Datasets

In [44]:
# ============================================================
# CELL 21 — EXPORT FINAL PREPARED DATASETS
# ============================================================

import os

print("=" * 70)
print("EXPORT FINAL PREPARED DATASETS")
print("=" * 70)

# 1. Kiểm tra các dataframe đầu vào
print(f"Original records        : {len(df):,}")
print(f"Deduplicated records    : {len(df_deduplicated):,}")
print(f"Genome-level records    : {len(df_genome_level):,}")
print(f"Genome Manifest records : {len(df_genome_manifest):,}")

# 2. Tạo thư mục output
OUTPUT_DIR = "/content/processed_data"
os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

# 3. Định nghĩa đường dẫn output

CLEANED_RECORDS_PATH = os.path.join(
    OUTPUT_DIR,
    "amr_cleaned_records.csv"
)

GENOME_LEVEL_PATH = os.path.join(
    OUTPUT_DIR,
    "genome_level_dataset.csv"
)

GENOME_MANIFEST_PATH = os.path.join(
    OUTPUT_DIR,
    "genome_manifest.csv"
)

# ------------------------------------------------------------
# 4. Export record-level dataset sau deduplication
# ------------------------------------------------------------

df_deduplicated.to_csv(
    CLEANED_RECORDS_PATH,
    index=False
)

# ------------------------------------------------------------
# 5. Export genome-level dataset
# ------------------------------------------------------------

df_genome_level.to_csv(
    GENOME_LEVEL_PATH,
    index=False
)

# 6. Export genome manifest

df_genome_manifest.to_csv(
    GENOME_MANIFEST_PATH,
    index=False
)


# 7. Kiểm tra file đã export
print("\nExported files:\n")

exported_files = [
    CLEANED_RECORDS_PATH,
    GENOME_LEVEL_PATH,
    GENOME_MANIFEST_PATH
]

for file_path in exported_files:

    file_size_mb = (
        os.path.getsize(file_path) / 1024**2
    )

    print(
        f"{os.path.basename(file_path):30s} "
        f"{file_size_mb:.2f} MB"
    )

# ------------------------------------------------------------
# 8. FINAL INTEGRITY CHECK
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("FINAL DATA PREPARATION SUMMARY")
print("=" * 70)

print(f"Original records              : {len(df):,}")
print(f"Exact duplicates removed      : {len(df) - len(df_deduplicated):,}")
print(f"Cleaned record-level dataset  : {len(df_deduplicated):,}")
print(f"Final unique genomes          : {len(df_genome_level):,}")
print(f"Genome manifest records       : {len(df_genome_manifest):,}")

print("\n" + "=" * 70)
print("DATA PREPARATION COMPLETED SUCCESSFULLY")
print("=" * 70)

print(
    "\nThe final datasets are ready for "
    "genome sequence retrieval and genomic feature engineering."
)

EXPORT FINAL PREPARED DATASETS
Original records        : 4,328
Deduplicated records    : 4,284
Genome-level records    : 4,270
Genome Manifest records : 4,270

Exported files:

amr_cleaned_records.csv        0.59 MB
genome_level_dataset.csv       0.63 MB
genome_manifest.csv            0.33 MB

FINAL DATA PREPARATION SUMMARY
Original records              : 4,328
Exact duplicates removed      : 44
Cleaned record-level dataset  : 4,284
Final unique genomes          : 4,270
Genome manifest records       : 4,270

DATA PREPARATION COMPLETED SUCCESSFULLY

The final datasets are ready for genome sequence retrieval and genomic feature engineering.
